<a href="https://colab.research.google.com/github/dsci3d/messyverse/blob/main/notebooks/01_isbn-open-library.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="In Colab oeffnen"/></a>

# ISBN-Liste anreichern -- ueber die Open Library

Die Bibliothek hat eine Liste von ISBNs (`katalog/isbns.txt`), aber zu vielen Titeln fehlen Autor und Erscheinungsjahr. Die **Open Library** ist ein frei zugaenglicher Web-Katalog fuer Buchmetadaten. Dein Auftrag: zu jeder ISBN Titel, Autor und Jahr holen und eine Tabelle bauen.

Du schreibst keinen Code selbst. Du schaust die Lage an, sagst deinem KI-Assistenten praezise, was du willst, fuegst seinen Code ein, fuehrst ihn aus und **pruefst gegen das Universum**. Wenn etwas klemmt, gibst du die Fehlermeldung an die KI zurueck -- du reparierst nichts von Hand.

In [ ]:
# SETUP (Black Box -- einmal ausfuehren). Holt deine Arbeitskopie des Uebungsuniversums.
import os, shutil, subprocess
ZIEL = "/content/messyverse"
if os.path.exists(ZIEL):
    shutil.rmtree(ZIEL)              # idempotent: alte Kopie weg, dann frisch klonen
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                "https://github.com/dsci3d/messyverse.git", ZIEL], check=True)
os.chdir(ZIEL)
anzahl = sum(len(dateien) for _, _, dateien in os.walk(ZIEL) if ".git" not in _)
print(f"Arbeitskopie: {anzahl} Dateien unter {ZIEL}")

## Wir arbeiten gegen gespeicherte Antworten (fixtures-first)

Damit die Uebung fuer alle gleich und ohne Wartezeit laeuft, liegen die Antworten der Open Library schon gespeichert in `api-fixtures/openlibrary/` -- eine JSON-Datei pro ISBN. Der **echte Live-Aufruf** ist der Bonus am Ende; er zeigt eine typische Stolperstelle (Weiterleitung). Der Vertrag (welche Felder, welcher Endpoint) steht in `api-fixtures/openlibrary/README.md`.

## Schritt 1 -- die Lage ansehen

Sieh dir zuerst an, womit du arbeitest: die ISBN-Liste und eine einzelne gespeicherte Antwort. Lass dir das von deiner KI zeigen, oder fuehre die naechste Zelle aus.

In [ ]:
print(open("katalog/isbns.txt").read())
import json
beispiel = json.load(open("api-fixtures/openlibrary/9783518111383.json"))
print(json.dumps(beispiel, ensure_ascii=False, indent=2)[:600])

## Schritt 2 -- die KI prompten

Formuliere deinen Auftrag an den Assistenten. Ein brauchbarer Prompt nennt die Datenquelle, das Soll-Ergebnis und das Ausgabeformat. Zum Beispiel:

> *Lies `katalog/isbns.txt`. Lade fuer jede ISBN die Datei `api-fixtures/openlibrary/<isbn>.json`. Zieh aus dem Objekt unter dem Schluessel `ISBN:<isbn>` den `title`, die Namen aus `authors` und `publish_date`. Baue ein dict namens `ergebnis`: ISBN -> {'titel':..., 'autoren':[...], 'jahr':...}. Gib zum Schluss eine Tabelle aus.*

Wichtig: lass die KI das Ergebnis genau `ergebnis` nennen -- die Pruefzelle weiter unten sucht danach.

In [ ]:
# Code deines KI-Assistenten hier einfuegen und ausfuehren.
# Am Ende muss ein dict `ergebnis` existieren: ISBN -> {'titel':..., 'autoren':[...], 'jahr':...}


## Schritt 3 -- gegen das Universum pruefen (Black Box)

Diese Zelle vergleicht dein `ergebnis` mit dem hinterlegten Soll und nennt dir, **wo** es klemmt -- ohne die Loesung zu verraten. Nicht editieren, nur ausfuehren.

In [ ]:
import json
soll = json.load(open("loesungen/isbn_lookup.golden.json"))
try:
    ergebnis
except NameError:
    print("Es gibt noch kein `ergebnis`. Lass deinen Assistenten das dict bauen (siehe Schritt 2).")
else:
    fehlen = [i for i in soll if i not in ergebnis]
    titel_ab = [i for i in soll if i in ergebnis and ergebnis[i].get("titel") != soll[i]["titel"]]
    if not fehlen and not titel_ab:
        print(f"OK -- alle {len(soll)} Titel stimmen mit dem Universum ueberein.")
    else:
        if fehlen: print("Diese ISBNs fehlen in deinem Ergebnis:", fehlen)
        if titel_ab: print("Bei diesen ISBNs weicht der Titel ab:", titel_ab)

## Wenn es klemmt -- die Fehlerschleife

Lief der Code auf einen Fehler, oder meldet die Pruefzelle Abweichungen? Dann **nicht selbst reparieren**. Kopiere die Fehlermeldung oder die Abweichungs-Liste, gib sie deinem Assistenten zurueck und bitte um eine korrigierte Fassung. Oft fehlt die Behandlung eines Sonderfalls -- etwa ein Autorname in arabischer Schrift oder eine Ausgabe mit Uebersetzer in der Autorenliste.

## Bonus (live) -- die echte Open Library und die 302-Falle

Optional, mit Netz: lass deine KI denselben Abruf gegen den echten Endpoint bauen:

> *Ruf `https://openlibrary.org/api/books?bibkeys=ISBN:9783518111383&jscmd=data&format=json` ab und vergleiche das Ergebnis mit der gespeicherten Datei.*

Beobachte, was bei manchen Anfragen mit einer Weiterleitung (HTTP 302) passiert -- und wie die KI darauf reagieren muss. Mit Pausen abrufen; 429/Timeout sind erwartbar.